- ### Class & Struct

1. [Class](#class)

default: private

2. [Struct](#struct)

default: public

3. [Object](#object)

4. [Encapsulation](#encapsulation)

public: accessible inside + outside + by derived class

protected: accessible inside + by derived class

private: accessible inside


5. [Method](#method)

constructor/destructor

function overload

static function

defined outside class

6. ["this" pointer](#this-pointer)

object pointer: for non-static function only

7. [Inheritance](#inheritance)

8. [Polymorphism](#polymorphism)

virtual function & function overriding

9. [Smart pointer](#smart-pointer)

10. [Nested class](#nested-class)

---

- ### Class

In [2]:
# Setup for oneline command %%cpp
import os, tempfile, subprocess
from IPython.core.magic import register_cell_magic # type: ignore
import shlex

@register_cell_magic
def cpp(line, cell):
    """
    Usage:
    %%cpp -i "input for cin" -- arg1 arg2 ...
    """
    tokens = shlex.split(line)
    input_data = None
    run_args = []

    # Parse stdin input
    if "-i" in tokens:
        idx = tokens.index("-i")
        if idx + 1 < len(tokens):
            input_data = tokens[idx + 1]

    # Parse program arguments after --
    if "--" in tokens:
        idx = tokens.index("--")
        run_args = tokens[idx + 1:]

    # Write temp C++ file
    with tempfile.NamedTemporaryFile(suffix=".cpp", delete=False, mode="w") as tmp_cpp:
        tmp_cpp.write(cell)
        cpp_path = tmp_cpp.name
    exe_path = cpp_path[:-4] + ".exe"

    try:
        # Compile
        compile_proc = subprocess.run(
            ["g++", "-std=c++23", "-O2", "-Wall", cpp_path, "-o", exe_path],
            capture_output=True,
            text=True
        )
        if compile_proc.returncode != 0:
            print("❌ Compilation failed:\n", compile_proc.stderr)
            return

        # Run program
        run_proc = subprocess.run(
            [exe_path] + run_args,
            input=input_data,      # feed stdin here
            capture_output=True,
            text=True
        )
        if run_proc.stdout:
            print(run_proc.stdout, end="")
        if run_proc.stderr:
            print("⚠️ Runtime error:\n", run_proc.stderr)

    finally:
        for f in (cpp_path, exe_path):
            try: os.remove(f)
            except: pass

In [3]:
%%cpp
#include <iostream>
using namespace std;

class Person {
    public:
        string name;
        int age;

        // Constructor
        Person(string n, int a) : name(n), age(a) {}

        // Method
        void introduce() {
            cout << "Hello, my name is " << name << " and I am " << age << " years old." << endl;
        }
};

int main() {
    Person p1("Alice", 30); // create an object instance
    p1.introduce(); // call method on the object
    Person p2("Bob", 25);
    p2.introduce();
}

Hello, my name is Alice and I am 30 years old.
Hello, my name is Bob and I am 25 years old.


---

- ### Struct

In [4]:
%%cpp
#include <iostream>
#include <cmath>
using namespace std;

struct Point {
    double x;
    double y;
    Point(double x_val, double y_val) : x(x_val), y(y_val) {}
    double norm() {
        return sqrt(x * x + y * y);
    }
};

int main() {
    Point p(3.0, 4.0);
    cout << "p = (" << p.x << ", " << p.y << ")" << endl; // access public members x, y
    cout << "Norm of p = " << p.norm() << endl;
}

p = (3, 4)
Norm of p = 5


---

-  ### Object

In [5]:
%%cpp
#include <iostream>
using namespace std;

class Vec {
    public:
        double x, y;

        // Constructor
        Vec(double x_val, double y_val) : x(x_val), y(y_val) {};
        // Method to add two vectors
        Vec add(const Vec &other) {
            return Vec(x + other.x, y + other.y);
        }
        // Method to display vector
        void display() {
            cout << "Vec(" << x << ", " << y << ")" << endl;
        }
};

int main() {
    Vec v1(1.0, 2.0);
    Vec v2(3.0, 4.0);
    Vec v3 = v1.add(v2);
    v3.display();
}

Vec(4, 6)


---

- ### Encapsulation

**public/private**

In [6]:
%%cpp
#include <iostream>
using namespace std;

class Rectangle {
    private:
        double width;
        double height;
    public:
        // Constructor
        Rectangle(double w, double h): width(w), height(h) {};
        // Method to calculate area
        double area() {
            return width * height;
        }
        // Method to calculate perimeter
        double perimeter() {
            return 2 * (width + height);
        }
};

int main() {
    Rectangle rect(5.0, 3.0);
    cout << "Area: " << rect.area() << endl;
    cout << "Perimeter: " << rect.perimeter() << endl;
    return 0;
}


Area: 15
Perimeter: 16


**protected**

In [43]:
%%cpp
#include <iostream>
using namespace std;

class Base {
    protected:
        int base_value;
};

class Derived : public Base {
    public:
        void Value(int v) {
            this->base_value = v;
            cout << "Base value: " << base_value << endl;
        }
};

int main() {
    Derived obj;
    obj.Value(2);
}

Base value: 2


---

- ### Method

**constructor/destructor**

In [7]:
%%cpp
#include <iostream>
using namespace std;
 class Student {
    private:
        string name;
        int age;
    public:
        // Constructor
        Student(string n, int a) // constructor with member initializer list
        :name(n), age(a) //initializer list
        { //constructor body
            cout << "Constructor called for " << name << endl;
        }; 
        // Method to display student info
        void displayInfo() {
            cout << "Name: " << name << ", Age: " << age << endl;
        }
        // Destructor to free memory
        ~Student() 
        {
            cout << "Destructor called for " << name << endl;
        }
 };

 int main() {
    Student s1("Charlie", 20);
    s1.displayInfo();
    Student s2("Diana", 22);
    s2.displayInfo();
    return 0;
 }

Constructor called for Charlie
Name: Charlie, Age: 20
Constructor called for Diana
Name: Diana, Age: 22
Destructor called for Diana
Destructor called for Charlie


**function overload**

In [8]:
%%cpp
#include <iostream>
using namespace std;

class Vec {
    public:
        double x, y;

        // Constructor
        Vec(double x_val, double y_val) 
        {   // constructor with assignment
            x = x_val, y = y_val;
        }; 
        // Method to add two vectors
        Vec operator+(const Vec &other) { // overloading + operator
            return Vec(x + other.x, y + other.y);
        }
        // Method to minus two vectors
        Vec operator-(const Vec &other) { // overloading - operator
            return Vec(x - other.x, y - other.y);
        }
        // Method to display vector
        void display() {
            cout << "Vec(" << x << ", " << y << ")" << endl;
        }
};

int main() {
    Vec v1(1.0, 2.0);
    Vec v2(3.0, 4.0);
    Vec v3 = v1 + v2;
    Vec v4 = v1 - v2;
    v3.display();
    v4.display();
}

Vec(4, 6)
Vec(-2, -2)


**static function**

In [9]:
%%cpp
#include <iostream>
using namespace std;

class Vec {
    public:
        double x, y;

        // Constructor
        Vec(double x_val, double y_val) : x(x_val), y(y_val) {};
        // Method to add two vectors
        static Vec add(const Vec &a, const Vec &b) { // static function
            return Vec(a.x + b.x, a.y + b.y);
        }
        // Method to display vector
        void display() {
            cout << "Vec(" << x << ", " << y << ")" << endl;
        }
};

int main() {
    Vec v1(1.0, 2.0);
    Vec v2(3.0, 4.0);
    Vec v3 = Vec::add(v1, v2);
    v3.display();
}

Vec(4, 6)


**defined outside class**

In [10]:
%%cpp
#include <iostream>
#include <cmath>
using namespace std;

class Vector3D {
    double x, y, z;
    public:
    void set(double a, double b, double c);
    void display();
    double norm();        
};

void Vector3D::set(double a, double b, double c) { // classname::methodname
    x = a;
    y = b;
    z = c;
}

void Vector3D::display() {
    cout << "Vector3D(" << x << ", " << y << ", " << z << ")" << endl;
}

double Vector3D::norm() {
    return sqrt(x*x + y*y + z*z);
}

int main() {
    Vector3D vec;
    vec.set(1.0, 2.0, 3.0);
    vec.display();
    cout << vec.norm() << endl;
}


Vector3D(1, 2, 3)
3.74166


---

- ### "this" pointer

In [ ]:
%%cpp
#include <iostream>
using namespace std;

class Point {
    public:
    int x,y;
        void setXY(int t, int k) {
            this->x = t; // using this pointer
            (*this).y = k; // the same as above
        }
};
int main() {
    Point p;
    p.setXY(10, 5);
    cout << "Point: " << "(" << p.x <<"," << p.y << ")" << endl;
}

Point: (10,5)


In [32]:
%%cpp
#include <iostream>
using namespace std;

class Point {
    public:
    int x,y;
    Point& setX(int x) {
        this->x = x; 
        return *this; // returning *this for method chaining
    }
    Point& setY(int y) {
        this->y = y; 
        return *this; // returning *this for method chaining
    }
};
int main() {
    Point p;
    p.setX(10).setY(5).setX(6); // method chaining
    cout << "Point: " << "(" << p.x <<"," << p.y << ")" << endl;
}

Point: (6,5)


---

- ### Inheritance

In [12]:
%%cpp
#include <iostream>
using namespace std;

class Animal {
    public:
        void eat() {
            cout << "Animal eats" << endl;
        }
};

class Dog : public Animal { // inheritance
    public:
        void bark() {
            cout << "Dog barks" << endl;
        }
};

int main() {
    Dog d;
    d.eat(); // inherited method
    d.bark();
}

Animal eats
Dog barks


---

- ### Polymorphism

**virtual function & function overriding**

In [13]:
%%cpp
#include <iostream>
using namespace std;
class Base {
    public:
        virtual void show() { // virtual function
            cout << "Base class show()" << endl;
        }
};

class Derived1 : public Base {
    public:
        void show() override { // override base class method
            cout << "Derived1 class show()" << endl;
        }
};

class Derived2 : public Base {
    public:
        void show() override { // override base class method
            cout << "Derived2 class show()" << endl;
        }
};

int main() {
    Base* bptr = new Derived1();
    bptr->show(); // calls derived1 class show() due to virtual function
    bptr = new Derived2();
    bptr->show(); // calls derived2 class show()
}

Derived1 class show()
Derived2 class show()


---

- ### Smart pointer

**unique pointer**

In [14]:
%%cpp
#include <iostream>
#include <memory>
#include <string>
using namespace std;

class Person {
    private:
        unique_ptr<string> name; // smart pointer for automatic memory management
    public:
    Person(const string& n) : name(make_unique<string>(n)) {} // constructor
    void introduce() {
        cout << "Hello, my name is " << *name << endl;
    }
};

int main() {
    Person p("Eve");
    p.introduce();
}

Hello, my name is Eve


**shared pointer**

In [15]:
%%cpp
#include <iostream>
#include <memory>
using namespace std;

class A{
    shared_ptr<int> ptr;
    public:
    A(shared_ptr<int> p) : ptr(p) {
        cout << ++(*p) << endl;
    };
};

int main() {
    auto p = make_shared<int>(2);
    A a1(p);
    A a2(p);
}

3
4


**pointer to a class/object**

In [16]:
%%cpp
#include <iostream>
using namespace std;
class Counter {
    private:
        static int count; // static member variable
    public:
        Counter() {
            count++;
        }
        static int getCount() { // static member function
            return count;
        }
        int getInstanceCount() { // non-static member function
            return count;
        }
};

int Counter::count = 0; // initialize static member variable

int main() {
    Counter c1;
    Counter c2;
    Counter c3;
    Counter* c4 = new Counter(); // pointer to object
    cout << "Instance count from c4: " << c4->getInstanceCount() << endl;
    cout << "Number of Counter objects: " << Counter::getCount() << endl;
}

Instance count from c4: 4
Number of Counter objects: 4


---

- ### Nested class

In [37]:
%%cpp
#include <iostream>
using namespace std;
class Outer {
    public:
        int x;
        void  setX(int x) {
            this->x = x;
        }
        class inner {
            public:
                int y;
                void  setY(int y) {
                    this->y = y;
        }
    };
};

int main() {
    Outer o;
    o.setX(10);
    Outer::inner i; // create instance of inner class
    i.setY(5);
    cout << "Outer x: " << o.x << ", Inner y: " << i.y << endl;
}

Outer x: 10, Inner y: 5
